# Belge (3) yöntemine göre saldırgan içerik tespiti

Bu notebook sekiz CSV'yi birlikte temizler; Word2Vec'i yalnızca train splitinde öğrenir ve belgede karşılaştırılan **LSTM**, **CNN** ve önerilen **CNN → LSTM** modellerini eğitir. BERT en sonda isteğe bağlı ayrı karşılaştırmadır.

Overfitting korumaları: train-only Word2Vec/vocabulary, dondurulmuş embedding, token ve SpatialDropout, L2, label smoothing, sınıf ağırlığı, gradient clipping, EarlyStopping, ReduceLROnPlateau ve en iyi validation checkpointi.

## 0. Çalışma zamanını ayarlayın
Colab menüsünden **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin. Projenin ana klasörünü Google Drive'a yükleyin.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

# Klasör adı veya Drive konumu farklı olsa da github_repo'yu otomatik bul.
MY_DRIVE = Path('/content/drive/MyDrive')
repo_candidates = [
    path for path in MY_DRIVE.rglob('github_repo')
    if path.is_dir() and (path / 'pyproject.toml').exists()
]
if not repo_candidates:
    repo_candidates = [
        path.parent for path in MY_DRIVE.rglob('pyproject.toml')
        if (path.parent / 'veri_temizlemesi_ve_egitimi').is_dir()
    ]
if not repo_candidates:
    raise FileNotFoundError(
        "Drive içinde github_repo bulunamadı. Proje ana klasörünü MyDrive'a yükleyin."
    )

def raw_data_score(repo_path):
    parent = repo_path.parent
    names = ['train_1.csv', 'test_1.csv', 'train_2.csv', 'test_2.csv']
    return sum((parent / name).exists() for name in names) + 4 * (parent / 'veri setleri ve url').is_dir()

REPO_DIR = max(repo_candidates, key=raw_data_score)
PROJECT_ROOT = REPO_DIR.parent
os.chdir(REPO_DIR)

required = [
    PROJECT_ROOT / 'train_1.csv',
    PROJECT_ROOT / 'test_1.csv',
    PROJECT_ROOT / 'train_2.csv',
    PROJECT_ROOT / 'test_2.csv',
    PROJECT_ROOT / 'veri setleri ve url',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Eksik veri/dizin: ' + ', '.join(missing))
print('Çalışma klasörü:', Path.cwd())
print('Yeni ve eski veri kaynakları bulundu.')

In [ ]:
%pip install -q -e ".[colab]"

# Colab imajında TensorFlow bulunmuyorsa otomatik kur.
import importlib.util, subprocess, sys
if importlib.util.find_spec('tensorflow') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'tensorflow>=2.18,<3'])
print('Bağımlılıklar hazır.')

In [ ]:
# GPU görünmüyorsa devam etmeyin; çalışma zamanı türünü T4 GPU yapın.
!nvidia-smi

## 1. Belge (3) yöntemine göre veri temizleme
Küçük harf, URL/kullanıcı adı, noktalama/özel karakter, fazla boşluk ve Türkçe durak kelime temizliği uygulanır. `değil`, `yok`, `hayır` anlamı korumak için silinmez. Çelişkiler ve kopyalar ayıklanır; yakın varyasyonlar aynı splitte tutulur.

In [ ]:
!python veri_temizlemesi_ve_egitimi/02_model_egitimi_colab.py \
    --prepare-data \
    --prepare-plots \
    --models lstm,cnn,cnn_lstm \
    --dry-run

!python tools/audit_dataset.py

Kontrol çıktısında `overlap`, `group_overlap` ve `label_conflicts` değerlerinin tamamı **0** olmalıdır.

## 2. Word2Vec + LSTM/CNN/CNN-LSTM eğitimi
Word2Vec yalnızca train metinlerinde bir kez eğitilir. Üç model aynı kelime vektörlerini kullanır; model seçimi validation F1 ile yapılır.

In [ ]:
!python veri_temizlemesi_ve_egitimi/02_model_egitimi_colab.py \
    --models lstm,cnn,cnn_lstm \
    --word2vec-epochs 10 \
    --keras-epochs 30 \
    --keras-batch-size 64

Bellek hatası alırsanız önceki hücrede yalnızca `--keras-batch-size 64` değerini `32` yapıp tekrar çalıştırın.

In [ ]:
import pandas as pd
from IPython.display import display, Image

results = pd.read_csv('sonuclar/model_karsilastirma.csv')
columns = [
    'model', 'validation_accuracy', 'validation_precision',
    'validation_recall', 'validation_f1', 'test_accuracy',
    'test_precision', 'test_recall', 'test_f1'
]
display(results[[column for column in columns if column in results.columns]])

graph_paths = sorted({
    *Path('grafikler').rglob('*.png'),
    *Path('sonuclar').rglob('*.png'),
})
print(f'{len(graph_paths)} grafik bulundu.')
for image_path in graph_paths:
    print(image_path)
    display(Image(filename=str(image_path), width=1100))

Aşağıdaki isteğe bağlı hücre bütün PNG grafiklerini tek ZIP dosyasında toplar.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('belge3_tum_grafikler', 'zip', root_dir='.', base_dir='grafikler')
if Path('sonuclar/grafikler').exists():
    !zip -qr belge3_tum_grafikler.zip sonuclar/grafikler
print('Hazır:', archive)
# İndirmek istediğinizde başındaki # işaretini kaldırın:
# files.download('belge3_tum_grafikler.zip')

## 3. İsteğe bağlı BERT karşılaştırması
Bu bölüm ana belge deneyinden ayrıdır. Önce LSTM/CNN/CNN-LSTM sonuçlarını tamamlayın.

In [ ]:
!python veri_temizlemesi_ve_egitimi/02_model_egitimi_colab.py \
    --models bert \
    --result-dir sonuclar/bert \
    --model-dir modeller/bert \
    --bert-epochs 3 \
    --bert-batch-size 16

BERT bellek hatası verirse batch boyutunu `8` yapın. Bağlantı kesilirse aynı komuta `--resume-from-checkpoint` ekleyin. Test sonuçlarını yalnızca nihai model validation ile seçildikten sonra tezde raporlayın.